In [ ]:
from pathlib import Path
import json
import requests
import re

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / 'backend').is_dir() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'backend').is_dir():
    raise FileNotFoundError('Could not find the HireAI project root. Run this notebook from within the project directory.')
DATASET_PROCESSED = PROJECT_ROOT / "ai" / "datasets" / "processed"

TEST_FILE = DATASET_PROCESSED / "hireai_test_v2.jsonl"

RESULT_FILE = DATASET_PROCESSED / "qwen3_v2_baseline_results.json"

OLLAMA_URL = "http://localhost:11434/api/generate"
MODEL_NAME = "qwen3:1.7b"

print("Project root:", PROJECT_ROOT)
print("Test dataset:", TEST_FILE)
print("Model:", MODEL_NAME)

In [ ]:
test_data = []

with open(TEST_FILE, "r", encoding="utf-8") as f:
    for line in f:
        test_data.append(json.loads(line))

print("Test examples:", len(test_data))
print()

print("First example:")
print(json.dumps(test_data[0], indent=2, ensure_ascii=False))

In [ ]:
def build_prompt(resume_text):
    return f"""
You are HireAI's resume analysis model.

Extract structured information from the resume below.

Return ONLY valid JSON.
Do not include explanations.
Do not include markdown.
Do not include ```json.
Do not invent information that is not present in the resume.

Use exactly this structure:

{{
    "candidate": {{
        "name": null,
        "email": null,
        "phone": null
    }},
    "summary": null,
    "skills": [],
    "years_of_experience": null,
    "job_titles": [],
    "experience": [],
    "education": [],
    "projects": [],
    "certifications": []
}}

Rules:

- Use null when information is unavailable.
- Keep skills as a list of strings.
- Keep job_titles as a list of strings.
- Preserve information from the resume.
- Do not add technologies that are not mentioned.
- Do not guess missing dates.
- Do not guess the candidate's experience.
- Return JSON only.

RESUME:
{resume_text}
""".strip()

In [ ]:
def ask_ollama(prompt):
    response = requests.post(
        OLLAMA_URL,
        json={
            "model": MODEL_NAME,
            "prompt": prompt,
            "stream": False,
            "options": {
                "temperature": 0
            }
        },
        timeout=180
    )

    response.raise_for_status()

    data = response.json()

    return data["response"]

In [ ]:
def extract_json(text):
    text = text.strip()

    # Remove Qwen thinking blocks
    text = re.sub(
        r"<think>.*?</think>",
        "",
        text,
        flags=re.DOTALL
    ).strip()

    # Remove markdown fences
    text = text.replace("```json", "")
    text = text.replace("```", "")
    text = text.strip()

    # Find first JSON object
    start = text.find("{")
    end = text.rfind("}")

    if start == -1 or end == -1:
        raise ValueError("No JSON object found.")

    json_text = text[start:end + 1]

    return json.loads(json_text)

In [ ]:
EXPECTED_FIELDS = {
    "candidate",
    "summary",
    "skills",
    "years_of_experience",
    "job_titles",
    "experience",
    "education",
    "projects",
    "certifications"
}


def validate_output_schema(data):

    if not isinstance(data, dict):
        return False

    required_fields = {
        "candidate",
        "summary",
        "skills",
        "years_of_experience",
        "job_titles",
        "experience",
        "education",
        "projects",
        "certifications"
    }

    if not required_fields.issubset(data.keys()):
        return False

    candidate = data["candidate"]

    if not isinstance(candidate, dict):
        return False

    required_candidate_fields = {
        "name",
        "email",
        "phone"
    }

    if not required_candidate_fields.issubset(candidate.keys()):
        return False

    # These fields should be lists when present.
    # None is also accepted and will be normalized later.
    list_fields = [
        "skills",
        "job_titles",
        "experience",
        "education",
        "projects",
        "certifications"
    ]

    for field in list_fields:

        value = data[field]

        if value is not None and not isinstance(value, list):
            return False

    return True

In [ ]:
def normalize_output(data):

    data = dict(data)

    list_fields = [
        "skills",
        "job_titles",
        "experience",
        "education",
        "projects",
        "certifications"
    ]

    for field in list_fields:

        if data.get(field) is None:
            data[field] = []

    return data

In [ ]:
example = test_data[0]

resume_text = example["input"]

prompt = build_prompt(resume_text)

raw_response = ask_ollama(prompt)

print(raw_response)

In [ ]:
parsed_response = extract_json(raw_response)

print(
    json.dumps(
        parsed_response,
        indent=2,
        ensure_ascii=False
    )
)

print(
    "\nSchema valid:",
    validate_output_schema(parsed_response)
)

In [ ]:
normalized_response = normalize_output(parsed_response)

print(
    json.dumps(
        normalized_response,
        indent=2,
        ensure_ascii=False
    )
)

In [ ]:
for i in range(3):

    print("=" * 70)
    print(f"TEST {i + 1}")
    print("=" * 70)

    example = test_data[i]

    prompt = build_prompt(example["input"])

    raw_response = ask_ollama(prompt)

    try:
        parsed = extract_json(raw_response)
        normalized = normalize_output(parsed)

        print(
            json.dumps(
                normalized,
                indent=2,
                ensure_ascii=False
            )
        )

        print(
            "\nSchema valid:",
            validate_output_schema(normalized)
        )

    except Exception as e:
        print("ERROR:", e)

In [ ]:
def normalize_text(value):
    if value is None:
        return ""

    return str(value).strip().lower()


def normalize_list(values):
    if not isinstance(values, list):
        return set()

    return {
        normalize_text(v)
        for v in values
        if v is not None
    }


def calculate_field_scores(predicted, actual):

    scores = {}

    # Candidate name
    scores["candidate.name"] = (
        normalize_text(predicted["candidate"]["name"])
        ==
        normalize_text(actual["candidate"]["name"])
    )

    # Email
    scores["candidate.email"] = (
        normalize_text(predicted["candidate"]["email"])
        ==
        normalize_text(actual["candidate"]["email"])
    )

    # Phone
    scores["candidate.phone"] = (
        normalize_text(predicted["candidate"]["phone"])
        ==
        normalize_text(actual["candidate"]["phone"])
    )

    # Skills
    predicted_skills = normalize_list(predicted["skills"])
    actual_skills = normalize_list(actual["skills"])

    if actual_skills:
        scores["skills"] = (
            len(predicted_skills & actual_skills)
            / len(actual_skills)
        )
    else:
        scores["skills"] = 1.0

    # Years of experience
    predicted_years = predicted["years_of_experience"]
    actual_years = actual["years_of_experience"]

    scores["years_of_experience"] = (
        predicted_years == actual_years
    )

    # Job titles
    predicted_titles = normalize_list(predicted["job_titles"])
    actual_titles = normalize_list(actual["job_titles"])

    if actual_titles:
        scores["job_titles"] = (
            len(predicted_titles & actual_titles)
            / len(actual_titles)
        )
    else:
        scores["job_titles"] = 1.0

    return scores

In [ ]:
results = []

for index, example in enumerate(test_data):

    print(
        f"Processing {index + 1}/{len(test_data)}...",
        end="\r"
    )

    try:
        prompt = build_prompt(example["input"])

        raw_response = ask_ollama(prompt)

        predicted = extract_json(raw_response)

        predicted = normalize_output(predicted)

        schema_valid = validate_output_schema(predicted)

        if schema_valid:
            scores = calculate_field_scores(
                predicted,
                example["output"]
            )
        else:
            scores = {}

        results.append({
            "index": index,
            "schema_valid": schema_valid,
            "scores": scores,
            "predicted": predicted if schema_valid else None
        })

    except Exception as e:

        results.append({
            "index": index,
            "schema_valid": False,
            "scores": {},
            "predicted": None,
            "error": str(e)
        })

print()
print("Evaluation complete.")

In [ ]:
field_totals = {}
schema_valid_count = 0

for result in results:

    if result["schema_valid"]:
        schema_valid_count += 1

    for field, score in result["scores"].items():

        if field not in field_totals:
            field_totals[field] = []

        field_totals[field].append(score)


print("\n")
print("=" * 60)
print("QWEN3 1.7B — V2 BASELINE RESULTS")
print("=" * 60)

print(
    f"Schema validity: "
    f"{schema_valid_count / len(results) * 100:.2f}%"
)

print()

for field, values in field_totals.items():

    average = sum(values) / len(values)

    print(
        f"{field:<30}: "
        f"{average * 100:.2f}%"
    )

In [ ]:
with open(
    RESULT_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        {
            "model": MODEL_NAME,
            "dataset": "hireai_test_v2",
            "num_examples": len(test_data),
            "schema_valid_count": schema_valid_count,
            "field_scores": {
                field: sum(values) / len(values)
                for field, values in field_totals.items()
            },
            "results": results
        },
        f,
        indent=2,
        ensure_ascii=False
    )

print("Results saved to:")
print(RESULT_FILE)